# Tarea para el Hogar 04 (Ejecución Local)
## Optimización de Hiperparámetros en LightGBM

### 1. Overfitting the Public Leaderboard
Lectura recomendada: https://medium.com/hmif-itb/overfitting-the-leaderboard-da25172ac62e

### 2. Objetivos
* Aumentar la rentabilidad de la campaña de marketing de retención proactiva de clientes.
* Generar un mejor modelo optimizando sus hiperparámetros.
* Conceptual: investigar los más relevantes hiperparámetros de LightGBM.
* Ver y ejecutar un pipeline completo de optimización de hiperparámetros y puesta en producción en entorno local.

#### 2.1 Seteo del ambiente local
Configuración de directorios de trabajo y verificación/descarga del dataset localmente.

In [1]:
# Determinar la raíz del proyecto (dmeyf2026) de forma robusta
find_project_root <- function() {
  curr <- normalizePath(getwd(), winslash = "/")
  while (curr != "/" && curr != dirname(curr)) {
    if (basename(curr) == "dmeyf2026" || (dir.exists(file.path(curr, "src")) && dir.exists(file.path(curr, "src", "ensembles")))) {
      return(curr)
    }
    curr <- dirname(curr)
  }
  if (dir.exists(file.path(getwd(), "dmeyf2026"))) {
    return(normalizePath(file.path(getwd(), "dmeyf2026"), winslash = "/"))
  }
  if (dir.exists("/home/sectorial/data-mining/dmeyf2026")) {
    return("/home/sectorial/data-mining/dmeyf2026")
  }
  return(normalizePath(getwd(), winslash = "/"))
}

dir_base <- find_project_root()
setwd(dir_base)
cat("Raíz del proyecto (dmeyf2026):", dir_base, "\n")

# Configurar entorno para proxy y herramientas CLI (Kaggle)
Sys.setenv(
  http_proxy = "http://10.4.8.20:8080",
  https_proxy = "http://10.4.8.20:8080",
  HTTP_PROXY = "http://10.4.8.20:8080",
  HTTPS_PROXY = "http://10.4.8.20:8080",
  PATH = paste("/home/sectorial/anaconda3/bin", file.path(Sys.getenv("HOME"), ".local/bin"), Sys.getenv("PATH"), sep = ":")
)

# Crear carpetas locales para datasets y experimentos dentro de dmeyf2026
dir.create(file.path(dir_base, "datasets"), showWarnings = FALSE, recursive = TRUE)
dir.create(file.path(dir_base, "exp"), showWarnings = FALSE, recursive = TRUE)

# Descargar el dataset si no existe localmente en dmeyf2026/datasets
url_dataset <- "https://storage.googleapis.com/open-courses/utn2026-b40a/dataset_pequeno.csv"
archivo_dataset <- file.path(dir_base, "datasets", "dataset_pequeno.csv")

if (!file.exists(archivo_dataset)) {
  cat("Descargando dataset_pequeno.csv en", archivo_dataset, "...\n")
  download.file(url_dataset, destfile = archivo_dataset, mode = "wb")
  cat("Descarga completada exitosamente.\n")
} else {
  cat("El dataset ya se encuentra disponible en:", archivo_dataset, "\n")
}

Raíz del proyecto (dmeyf2026): /workspace/dmeyf2026 
El dataset ya se encuentra disponible en: /workspace/dmeyf2026/datasets/dataset_pequeno.csv 


### 2.2 Optimización de Hiperparámetros
#### 2.2.1 Inicio y limpieza de memoria

In [2]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Thu Aug 27 16:37:43 2026"

In [3]:
# limpio la memoria
rm(list = ls(all.names = TRUE)) # remove all objects
gc(full = TRUE, verbose = FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,661963,35.4,1454520,77.7,1454520,77.7
Vcells,1232926,9.5,8388608,64.0,1975054,15.1


#### 2.2.2 Carga de Librerías

In [4]:
# Cargo librerías requeridas (instalando automáticamente si no existen)
if (!require("data.table")) install.packages("data.table", repos = "https://cloud.r-project.org")
require("data.table")

if (!require("parallel")) install.packages("parallel", repos = "https://cloud.r-project.org")
require("parallel")

if (!require("primes")) install.packages("primes", repos = "https://cloud.r-project.org")
require("primes")

if (!require("utils")) install.packages("utils", repos = "https://cloud.r-project.org")
require("utils")

if (!require("rlist")) install.packages("rlist", repos = "https://cloud.r-project.org")
require("rlist")

if (!require("yaml")) install.packages("yaml", repos = "https://cloud.r-project.org")
require("yaml")

if (!require("lightgbm")) install.packages("lightgbm", repos = "https://cloud.r-project.org")
require("lightgbm")

Loading required package: data.table

Loading required package: parallel

Loading required package: primes

Loading required package: rlist

Loading required package: yaml

Loading required package: lightgbm



#### 2.2.3 Definición de Parámetros
Aquí se debe configurar su semilla primigenia y el número de experimento.

In [5]:
PARAM <- list()
PARAM$experimento <- '5940_01'
PARAM$semilla_primigenia <- 115879 # Reemplazar con su semilla primigenia si corresponde

PARAM$kaggle$competencia <- "utn-2026-inicial"
PARAM$kaggle$cortes <- seq(9000, 12000, by = 500)

# Estrategia de entrenamiento: undersampling de los CONTINUA
# 0.5 toma el 50% de los CONTINUA y 100% de BAJA+1 y BAJA+2
PARAM$trainingstrategy$undersampling <- 0.5

# Folds para Cross-Validation
PARAM$hyperparametertuning$xval_folds <- 5

# Parámetros fijos del LightGBM que se pisarán con la parte variable de la búsqueda
PARAM$lgbm$param_fijos <- list(
  boosting = "gbdt",
  objective = "binary",
  metric = "auc",
  first_metric_only = FALSE,
  boost_from_average = TRUE,
  feature_pre_filter = FALSE,
  force_row_wise = TRUE, # para reducir warnings
  verbosity = -100,

  seed = PARAM$semilla_primigenia,

  max_depth = -1L, # -1 significa no limitar la profundidad
  min_gain_to_split = 0,
  min_sum_hessian_in_leaf = 0.001,
  lambda_l1 = 0.0,
  lambda_l2 = 0.0,
  max_bin = 31L,

  bagging_fraction = 1.0,
  pos_bagging_fraction = 1.0,
  neg_bagging_fraction = 1.0,
  is_unbalance = FALSE,
  scale_pos_weight = 1.0,

  drop_rate = 0.1,
  max_drop = 50,
  skip_drop = 0.5,

  extra_trees = FALSE,

  num_iterations = 100,
  learning_rate = 0.10,
  feature_fraction = 1.0,
  num_leaves = 32,
  min_data_in_leaf = 20
)

#### 2.2.4 Preprocesamiento y Creación del Dataset de Entrenamiento

In [6]:
# Determinar la raíz del proyecto (dmeyf2026) de forma robusta
find_project_root <- function() {
  curr <- normalizePath(getwd(), winslash = "/")
  while (curr != "/" && curr != dirname(curr)) {
    if (basename(curr) == "dmeyf2026" || (dir.exists(file.path(curr, "src")) && dir.exists(file.path(curr, "src", "ensembles")))) {
      return(curr)
    }
    curr <- dirname(curr)
  }
  if (dir.exists(file.path(getwd(), "dmeyf2026"))) {
    return(normalizePath(file.path(getwd(), "dmeyf2026"), winslash = "/"))
  }
  if (dir.exists("/home/sectorial/data-mining/dmeyf2026")) {
    return("/home/sectorial/data-mining/dmeyf2026")
  }
  return(normalizePath(getwd(), winslash = "/"))
}

dir_base <- find_project_root()
experimento_folder <- paste0("HT", PARAM$experimento)
dir_exp <- file.path(dir_base, "exp", experimento_folder)
dir.create(dir_exp, showWarnings = FALSE, recursive = TRUE)
setwd(dir_exp)
cat("Directorio de trabajo actual:", getwd(), "\n")

Directorio de trabajo actual: /workspace/dmeyf2026/exp/HT5940_01 


In [7]:
# Lectura del dataset local
archivo_dataset <- file.path(dir_base, "datasets", "dataset_pequeno.csv")
dataset <- fread(archivo_dataset)

# Filtrar mes de entrenamiento (202107)
dataset_train <- dataset[foto_mes %in% c(202107)]

# Paso la clase a binaria {0, 1}: BAJA+1 y BAJA+2 son 1, CONTINUA es 0
dataset_train[, clase01 := ifelse(clase_ternaria %in% c("BAJA+2", "BAJA+1"), 1L, 0L)]

# Undersampling de los CONTINUA usando la semilla primigenia
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset_train[, azar := runif(nrow(dataset_train))]
dataset_train[, training := 0L]
dataset_train[
  foto_mes %in% c(202107) &
    (azar <= PARAM$trainingstrategy$undersampling | clase_ternaria %in% c("BAJA+1", "BAJA+2")),
  training := 1L
]

# Campos a utilizar
campos_buenos <- setdiff(
  colnames(dataset_train),
  c("clase_ternaria", "clase01", "azar", "training")
)

# Crear estructura lgb.Dataset
dtrain <- lgb.Dataset(
  data = data.matrix(dataset_train[training == 1L, campos_buenos, with = FALSE]),
  label = dataset_train[training == 1L, clase01],
  free_raw_data = FALSE
)

cat("Registros para CV (con undersampling):", nrow(dtrain), "| Columnas:", ncol(dtrain), "\n")

Registros para CV (con undersampling): 83081 | Columnas: 154 


#### 2.2.5 Configuración y Ejecución del Grid Search

In [8]:
# Función que estima el AUC en Cross-Validation de 5 folds para un vector de hiperparámetros 'x'
Estimar_AUC_lightgbm <- function(x) {
  # x pisa o agrega a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM con validación cruzada
  modelocv <- lgb.cv(
    data = dtrain,
    nfold = PARAM$hyperparametertuning$xval_folds,
    stratified = TRUE,
    param = param_completo
  )

  # obtengo la mejor métrica (AUC)
  AUC <- modelocv$best_score

  # libero memoria
  rm(modelocv)
  gc(full = TRUE, verbose = FALSE)

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " AUC ", AUC
  )

  return(AUC)
}

In [9]:
# Definir el producto cartesiano de hiperparámetros a explorar
# Aquí usted puede experimentar agregando/modificando los valores y parámetros

# Configuración por defecto
# tb_nueva <- CJ(
#   num_iterations = c(100, 500, 1000),
#   learning_rate = c(0.02, 0.05, 0.1),
#   num_leaves = c(10, 50, 100, 500)
# )

tb_nueva <- CJ(
  learning_rate    = c(0.02, 0.05),
  num_leaves       = c(50, 80, 120),
  min_data_in_leaf = c(200, 500),
  feature_fraction = c(0.4, 0.7),
  num_iterations   = c(300, 600, 1000)
)

cat("Cantidad total de combinaciones a evaluar:", nrow(tb_nueva), "\n")
tb_nueva

Cantidad total de combinaciones a evaluar: 72 


learning_rate,num_leaves,min_data_in_leaf,feature_fraction,num_iterations
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
0.02,50,200,0.4,300
0.02,50,200,0.4,600
0.02,50,200,0.4,1000
0.02,50,200,0.7,300
0.02,50,200,0.7,600
0.02,50,200,0.7,1000
0.02,50,500,0.4,300
0.02,50,500,0.4,600
0.02,50,500,0.4,1000


In [ ]:
# Registro a registro calculo la AUC en Cross-Validation
tb_nueva[, AUC := Estimar_AUC_lightgbm(list(.SD)), by = 1:nrow(tb_nueva)]

# Mostrar resultados ordenados
setorder(tb_nueva, -AUC)
tb_nueva

Thu Aug 27 16:38:15 2026  list(learning_rate = 0.02, num_leaves = 50, min_data_in_leaf = 200, feature_fraction = 0.4, num_iterations = 300) AUC 0.925893460664036

Thu Aug 27 16:38:42 2026  list(learning_rate = 0.02, num_leaves = 50, min_data_in_leaf = 200, feature_fraction = 0.4, num_iterations = 600) AUC 0.924973964875888

Thu Aug 27 16:39:07 2026  list(learning_rate = 0.02, num_leaves = 50, min_data_in_leaf = 200, feature_fraction = 0.4, num_iterations = 1000) AUC 0.926631616313201

Thu Aug 27 16:39:36 2026  list(learning_rate = 0.02, num_leaves = 50, min_data_in_leaf = 200, feature_fraction = 0.7, num_iterations = 300) AUC 0.925268741275167

Thu Aug 27 16:40:09 2026  list(learning_rate = 0.02, num_leaves = 50, min_data_in_leaf = 200, feature_fraction = 0.7, num_iterations = 600) AUC 0.925317751186717

Thu Aug 27 16:40:34 2026  list(learning_rate = 0.02, num_leaves = 50, min_data_in_leaf = 200, feature_fraction = 0.7, num_iterations = 1000) AUC 0.925873578603952

Thu Aug 27 16:41:01 

In [ ]:
# Grabar tabla con los resultados del Grid Search
fwrite(tb_nueva,
  file = "tb_grid_search_01.txt",
  sep = "\t",
  append = TRUE
)

# Guardar los mejores hiperparámetros en PARAM
PARAM$out$lgbm$AUC <- tb_nueva[1, AUC]
PARAM$out$lgbm$mejores_hiperparametros <- as.list(tb_nueva[1])
PARAM$out$lgbm$mejores_hiperparametros$AUC <- NULL

cat("Mejor AUC obtenida:", PARAM$out$lgbm$AUC, "\n")
cat("Mejores hiperparámetros:\n")
print(PARAM$out$lgbm$mejores_hiperparametros)

# Guardar PARAM en YAML
write_yaml(PARAM, file = "PARAM.yml")

## 2.3 Producción / Final Training
Construyo el modelo final con los mejores hiperparámetros encontrados, entrenando sobre TODOS los datos de `202107` (sin undersampling) y ajustando `min_data_in_leaf`.

In [ ]:
# Carpeta de trabajo para el modelo final
experimento <- paste0("exp", PARAM$experimento)
dir_final <- file.path(dir_base, "exp", experimento)
dir.create(dir_final, showWarnings = FALSE, recursive = TRUE)
setwd(dir_final)
cat("Directorio de trabajo Producción:", getwd(), "\n")

In [ ]:
# Liberar memoria previa del Grid Search
if (exists("dtrain")) rm(dtrain)
if (exists("dataset_train")) rm(dataset_train)
gc(full = TRUE, verbose = FALSE)

# Preparación del dataset completo para Final Training
dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]
dataset_train <- dataset[foto_mes %in% c(202107)]

# Dataset de LightGBM sobre el 100% de datos de 202107 (sin undersampling)
dtrain_final <- lgb.Dataset(
  data = data.matrix(dataset_train[, campos_buenos, with = FALSE]),
  label = dataset_train[, clase01],
  free_raw_data = TRUE
)

# Liberar dataset_train de R ya que dtrain_final está creado
rm(dataset_train)
gc(full = TRUE, verbose = FALSE)

# Combinar parámetros fijos con los mejores encontrados
param_final <- modifyList(PARAM$lgbm$param_fijos, PARAM$out$lgbm$mejores_hiperparametros)

# Ajuste / Normalización y casteo explícito de tipos enteros para LightGBM C++
param_normalizado <- copy(param_final)
param_normalizado$min_data_in_leaf <- as.integer(round(as.numeric(param_final$min_data_in_leaf) / as.numeric(PARAM$trainingstrategy$undersampling)))
if (!is.null(param_normalizado$num_leaves)) param_normalizado$num_leaves <- as.integer(param_normalizado$num_leaves)
if (!is.null(param_normalizado$num_iterations)) param_normalizado$num_iterations <- as.integer(param_normalizado$num_iterations)
if (!is.null(param_normalizado$max_bin)) param_normalizado$max_bin <- as.integer(param_normalizado$max_bin)
if (!is.null(param_normalizado$max_depth)) param_normalizado$max_depth <- as.integer(param_normalizado$max_depth)
if (!is.null(param_normalizado$seed)) param_normalizado$seed <- as.integer(param_normalizado$seed)

cat("Hiperparámetros normalizados para Final Training:\n")
print(param_normalizado)

In [ ]:
# Asegurar memoria limpia antes de entrenar
gc(full = TRUE, verbose = FALSE)

# Entreno el modelo final
cat("Iniciando entrenamiento del modelo final...\n")
modelo_final <- lgb.train(
  data = dtrain_final,
  params = param_normalizado
)
cat("Entrenamiento completado exitosamente.\n")

# Imprimir y guardar importancia de variables
tb_importancia <- as.data.table(lgb.importance(modelo_final))
fwrite(tb_importancia, file = "impo.txt", sep = "\t")
cat("Top 10 variables más importantes:\n")
print(head(tb_importancia, 10))

# Grabar a disco el modelo entrenado
lgb.save(modelo_final, "modelo.txt")
cat("Modelo guardado en modelo.txt\n")

### 2.4 Scoring y Generación de Envíos a Kaggle
Aplico el modelo a los datos del futuro (`foto_mes == 202109`) y genero los archivos de corte.

In [ ]:
# Aplicar el modelo a los datos de 202109
dfuture <- dataset[foto_mes == 202109]
prediccion <- predict(
  modelo_final,
  data.matrix(dfuture[, campos_buenos, with = FALSE])
)

# Crear tabla de predicción
tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := prediccion]

# Grabar probabilidades
fwrite(tb_prediccion, file = "prediccion.txt", sep = "\t")
cat("Predicciones guardadas en prediccion.txt\n")

# Liberar memoria del modelo y datasets de entrenamiento
rm(dtrain_final, modelo_final)
gc(full = TRUE, verbose = FALSE)

In [ ]:
# Generar envíos para Kaggle para distintos puntos de corte
setorder(tb_prediccion, -prob)
dir.create("kaggle", showWarnings = FALSE)

kaggle_bin <- Sys.which("kaggle")
if (kaggle_bin == "") kaggle_bin <- "/home/sectorial/anaconda3/bin/kaggle"

for (envios in PARAM$kaggle$cortes) {
  tb_prediccion[, Predicted := 0L]
  tb_prediccion[1:envios, Predicted := 1L]

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file = archivo_kaggle,
    sep = ","
  )
  cat("Generado:", archivo_kaggle, "con", envios, "envíos positivos.\n")

  # Subida automática a Kaggle
  mensaje <- paste0("envios=", envios, " semilla=", PARAM$semilla_primigenia)
  linea <- paste0(
    kaggle_bin, " competitions submit -c ", PARAM$kaggle$competencia,
    " -f ", archivo_kaggle,
    " -m '", mensaje, "'"
  )
  tryCatch({
    salida <- system(linea, intern = TRUE)
    cat(paste(salida, collapse = "\n"), "\n")
    Sys.sleep(10)
  }, error = function(e) {
    cat("Nota: No se pudo realizar el submit automático:", conditionMessage(e), "\n")
  })
}

# Guardar parámetros finales y timestamp
write_yaml(PARAM, file = "PARAM.yml")
format(Sys.time(), "%a %b %d %X %Y")

### 2.5 Registro en la Google Sheet Colaborativa
Cargue los resultados de su corrida en la Google Sheet Colaborativa, en la hoja **`TareaHogar-04`**:
* Número de experimento
* Semilla utilizada
* Mejores hiperparámetros encontrados y su AUC en Cross-Validation
* Ganancias en Kaggle para los distintos puntos de corte (9.000 a 12.000)